
#**PROYECTO FINAL BIG DATA**

*Identificación de zonas críticas de criminalidad para apoyar la toma de decisiones en seguridad ciudadana*


**Descripción del problema** 

La criminalidad representa uno de los principales retos para la seguridad ciudadana en Colombia, afectando la calidad de vida de la población y generando impactos sociales y económicos significativos. Aunque las entidades gubernamentales y organismos de seguridad recopilan continuamente grandes volúmenes de información sobre hechos delictivos en diferentes municipios y departamentos del país, estos datos suelen utilizarse principalmente para reportes históricos y análisis descriptivos, limitando su aprovechamiento para la toma de decisiones estratégicas.

La identificación oportuna de zonas con alta concentración de delitos constituye un desafío debido al volumen, variedad y crecimiento constante de los registros disponibles. Esta situación dificulta detectar patrones geográficos y temporales que permitan anticipar comportamientos delictivos, priorizar territorios de intervención y optimizar la asignación de recursos destinados a la prevención y control del crimen.

En este contexto, surge la necesidad de implementar una solución basada en tecnologías Big Data que permita procesar y analizar grandes cantidades de datos de criminalidad de manera eficiente. A través de técnicas de análisis de datos, visualización geoespacial y modelos predictivos, se busca identificar zonas críticas de criminalidad y generar información de valor para apoyar la toma de decisiones en seguridad ciudadana. Esto permitirá a las autoridades contar con herramientas analíticas que faciliten la focalización de estrategias preventivas, la distribución eficiente de recursos y el fortalecimiento de las acciones orientadas a reducir los índices delictivos en las áreas de mayor riesgo.

*Problema central del proyecto*

¿Cómo utilizar grandes volúmenes de datos históricos de criminalidad para identificar zonas críticas y generar información que apoye la toma de decisiones en materia de seguridad ciudadana y prevención del delito en Colombia?

**Objetivo General**

Desarrollar una plataforma Big Data para identificar zonas críticas de criminalidad en Colombia mediante el análisis de más de 2 millones de registros históricos, permitiendo apoyar la toma de decisiones en seguridad ciudadana mediante análisis descriptivos y modelos predictivos.

**Objetivos Específicos**

Centralizar y almacenar grandes volúmenes de datos de criminalidad.
Automatizar los procesos de ingesta, limpieza y transformación de datos.
Identificar patrones geográficos y temporales asociados a la ocurrencia de delitos.
Construir modelos predictivos que permitan estimar comportamientos futuros de criminalidad.
Generar visualizaciones interactivas para facilitar la toma de decisiones.
Apoyar la asignación eficiente de recursos de vigilancia y prevención.

**RELACIÓN BENEFICIO / COSTE**

**Arquitectura de Solución**
Flujo propuesto:

CSV (Kaggle)

↓

Azure Data Lake

↓

Databricks Auto Loader

↓

BRONZE

Datos Crudos

↓

SILVER

Limpieza y Enriquecimiento

↓

GOLD

KPIs y Zonas Críticas

↓

Machine Learning

↓

Power BI

**Explicación Técnica**

*Fuente de Datos*

Dataset histórico de criminalidad en Colombia proveniente de Kaggle.

*Azure Data Lake Storage*

Repositorio central para almacenar datos históricos y archivos de entrada.

*Azure Databricks*

Motor principal para procesamiento distribuido mediante Apache Spark.

*Bronze Layer*

Almacenamiento de datos originales sin modificaciones.

*Silver Layer*

Procesos de limpieza, validación y enriquecimiento.

*Gold Layer*

Generación de indicadores de negocio y datasets analíticos.

*Machine Learning*

Modelos predictivos para estimar tendencias criminales.

*Power BI*

Visualización de resultados y monitoreo de indicadores.

**PIPELINE DE INGESTA DE DATOS**

Automatización de Ingesta

La solución implementa pipelines automáticos encargados de cargar periódicamente la información histórica desde las fuentes de datos hacia la arquitectura Medallion.

Etapas del Pipeline
Captura de datos.
Validación de estructura.
Almacenamiento Bronze.
Transformaciones Silver.
Agregaciones Gold.
Entrenamiento de modelos.
Actualización de dashboards.
Estrategia Medallion
Bronze Layer

Datos originales provenientes de Kaggle.

Características:

Sin transformaciones.
Conservación de la fuente original.
Auditoría de datos.
Silver Layer

Transformaciones realizadas:

Eliminación de duplicados.
Tratamiento de valores nulos.
Conversión de fechas.
Estandarización de municipios y departamentos.
Generación de variables temporales.

Columnas derivadas:

Año
Mes
Día
Trimestre
Región
Gold Layer

Tablas analíticas finales:

Delitos por Departamento

Permite identificar territorios con mayor incidencia.

Delitos por Municipio

Identificación de zonas críticas.

Delitos por Tipo

Análisis de modalidades delictivas.

Tendencias Temporales

Comportamiento histórico de la criminalidad.

Workflows

Los workflows de Databricks automatizan:

Ingesta diaria.
Transformaciones.
Actualización de tablas Gold.
Entrenamiento de modelos predictivos.
Publicación de resultados.



In [0]:
from pyspark.sql import functions as F
import pandas as pd
import os

# Ruta relativa del archivo CSV en el workspace (mismo directorio que el notebook)
csv_path = "v_delitos.csv"

# Verificar si el archivo existe en el directorio actual
if os.path.exists(csv_path):
    print(f"✓ Archivo encontrado: {csv_path}")
    
    # Leer el CSV con pandas
    pdf = pd.read_csv(csv_path)
    print(f"✓ Datos cargados con pandas: {len(pdf)} registros")
    
    # Convertir a Spark DataFrame
    bronze_df = spark.createDataFrame(pdf)
    print(f"✓ Convertido a Spark DataFrame")
    
else:
    # Intentar con la ruta absoluta del workspace
    print("Archivo no encontrado en directorio actual, intentando dbutils...")
    
    # Usar dbutils para listar archivos
    files = dbutils.fs.ls("/Big_data_TF/Trabajo FInal/")
    print("Archivos encontrados:")
    for f in files:
        print(f"  - {f.name}")
    
    # Leer usando dbutils si está disponible
    bronze_df = (
        spark.read
        .option("header","true")
        .option("inferSchema","true")
        .csv("dbfs:/Big_data_TF/Trabajo FInal/v_delitos.csv")
    )

# Guardar en la capa Bronze
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "criminalidad.seguridad_ciudadana.bronze_delitos"
    )

In [0]:
display(
    spark.table(
        "criminalidad.seguridad_ciudadana.bronze_delitos"
    )
)

In [0]:
from pyspark.sql.functions import *

In [0]:
bronze = spark.table(
    "criminalidad.seguridad_ciudadana.bronze_delitos"
)

In [0]:
silver = (
    bronze
    .dropDuplicates()
    .withColumn(
        "fecha",
        to_date("fecha")
    )
    .withColumn(
        "anio",
        year("fecha")
    )
    .withColumn(
        "mes",
        month("fecha")
    )
    .withColumn(
        "trimestre",
        quarter("fecha")
    )
)

In [0]:
silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "criminalidad.seguridad_ciudadana.silver_delitos"
    )

In [0]:
silver = spark.table(
    "criminalidad.seguridad_ciudadana.silver_delitos"
)

gold_departamento = (
    silver
    .groupBy("departamento")
    .agg(
        sum("cantidad").alias("total_delitos")
    )
)

In [0]:
gold_departamento.write \
.mode("overwrite") \
.saveAsTable(
"criminalidad.seguridad_ciudadana.gold_delitos_departamento"
)

In [0]:
gold_municipio = (
    silver
    .groupBy(
        "departamento",
        "municipio"
    )
    .agg(
        sum("cantidad").alias("total_delitos")
    )
)

In [0]:
gold_tipo = (
    silver
    .groupBy("tipo")
    .agg(
        sum("cantidad").alias("total_delitos")
    )
)

In [0]:
gold_genero = (
    silver
    .groupBy("genero")
    .agg(
        sum("cantidad").alias("total_delitos")
    )
)

In [0]:
gold_edad = (
    silver
    .groupBy("rango_edad")
    .agg(
        sum("cantidad").alias("total_delitos")
    )
)

In [0]:
zonas_criticas = (
    silver
    .groupBy(
        "departamento",
        "municipio"
    )
    .agg(
        sum("cantidad").alias("total_delitos")
    )
    .orderBy(
        col("total_delitos").desc()
    )
)

In [0]:
zonas_criticas.write \
.mode("overwrite") \
.saveAsTable(
"criminalidad.seguridad_ciudadana.gold_zonas_criticas"
)

In [0]:
from pyspark.ml.feature import StringIndexer

In [0]:
silver = spark.table(
    "criminalidad.seguridad_ciudadana.silver_delitos"
)

In [0]:
departamento_index = StringIndexer(
    inputCol="departamento",
    outputCol="departamento_idx"
)

municipio_index = StringIndexer(
    inputCol="municipio",
    outputCol="municipio_idx"
)

tipo_index = StringIndexer(
    inputCol="tipo",
    outputCol="tipo_idx"
)


In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

In [0]:
assembler = VectorAssembler(
    inputCols=[
        "anio",
        "mes",
        "departamento_idx",
        "municipio_idx",
        "tipo_idx"
    ],
    outputCol="features"
)

In [0]:
rf = RandomForestRegressor(
    labelCol="cantidad",
    featuresCol="features",
    numTrees=100
)

In [0]:
pipeline = Pipeline(
    stages=[
        departamento_index,
        municipio_index,
        tipo_index,
        assembler,
        rf
    ]
)

In [0]:
silver = spark.table(
    "criminalidad.seguridad_ciudadana.silver_delitos"
)

In [0]:
from pyspark.sql import functions as F

dataset = (
    silver
    .groupBy(
        "departamento",
        "municipio",
        "anio",
        "mes"
    )
    .agg(
        F.sum("cantidad").alias("total_delitos")
    )
)

In [0]:
display(dataset)

In [0]:
from pyspark.ml.feature import StringIndexer

In [0]:
departamento_indexer = StringIndexer(
    inputCol="departamento",
    outputCol="departamento_idx",
    handleInvalid="keep"
)

municipio_indexer = StringIndexer(
    inputCol="municipio",
    outputCol="municipio_idx",
    handleInvalid="keep"
)

In [0]:
dataset_idx = departamento_indexer.fit(dataset).transform(dataset)

dataset_idx = municipio_indexer.fit(dataset_idx).transform(dataset_idx)

In [0]:
dataset.count()


In [0]:
display(dataset.limit(10))

In [0]:
dataset

In [0]:
from pyspark.sql import functions as F

dataset = (
    silver
    .groupBy(
        "cod_dane",
        "departamento",
        "municipio",
        "anio",
        "mes"
    )
    .agg(
        F.sum("cantidad").alias("total_delitos")
    )
)

In [0]:
display(dataset)

In [0]:
dataset = dataset.na.drop()

In [0]:
from pyspark.ml.feature import VectorAssembler

In [0]:
assembler = VectorAssembler(
    inputCols=[
        "anio",
        "mes",
        "cod_dane"
    ],
    outputCol="features"
)

In [0]:
dataset_final = assembler.transform(dataset)

In [0]:
train, test = dataset_final.randomSplit(
    [0.8, 0.2],
    seed=42
)

In [0]:
print("Train:", train.count())
print("Test:", test.count())

In [0]:
from pyspark.ml.regression import RandomForestRegressor

In [0]:
rf = RandomForestRegressor(
    labelCol="total_delitos",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    seed=42
)

In [0]:
model = rf.fit(train)

In [0]:
predicciones = model.transform(test)

In [0]:
display(
    predicciones.select(
        "departamento",
        "municipio",
        "anio",
        "mes",
        "total_delitos",
        "prediction"
    )
)

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

In [0]:
rmse = RegressionEvaluator(
    labelCol="total_delitos",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(predicciones)

print("RMSE:", rmse)

In [0]:
mae = RegressionEvaluator(
    labelCol="total_delitos",
    predictionCol="prediction",
    metricName="mae"
).evaluate(predicciones)

print("MAE:", mae)

In [0]:
r2 = RegressionEvaluator(
    labelCol="total_delitos",
    predictionCol="prediction",
    metricName="r2"
).evaluate(predicciones)

print("R2:", r2)

In [0]:
import pandas as pd

In [0]:
importances = model.featureImportances

In [0]:
feature_names = [
    "anio",
    "mes",
    "cod_dane"
]

In [0]:
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": list(importances)
})

importance_df.sort_values(
    by="importance",
    ascending=False
)

In [0]:
predicciones.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "criminalidad.seguridad_ciudadana.gold_predicciones"
    )

In [0]:
zonas_criticas = (
    silver
    .groupBy(
        "departamento",
        "municipio"
    )
    .agg(
        F.sum("cantidad").alias("total_delitos")
    )
    .orderBy(
        F.col("total_delitos").desc()
    )
)

In [0]:
zonas_criticas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "criminalidad.seguridad_ciudadana.gold_zonas_criticas"
    )

In [0]:
%sql
CREATE OR REPLACE VIEW criminalidad.seguridad_ciudadana.v_dashboard AS

SELECT
    departamento,
    municipio,
    SUM(cantidad) total_delitos
FROM criminalidad.seguridad_ciudadana.silver_delitos
GROUP BY
    departamento,
    municipio;

In [0]:
%sql
SELECT *
FROM criminalidad.seguridad_ciudadana.gold_zonas_criticas
ORDER BY total_delitos DESC
LIMIT 20;